# adk-polyglot-workshop - Hands-on Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JAICHANGPARK/adk-polyglot-workshop/blob/main/output/colab/workshop_starter.ipynb)

> **Topic**: Enterprise Multi-Agent SRE with Polyglot ADK

Welcome to the hands-on lab. Run each cell sequentially and fill in the `TODO` sections.

In [ ]:
# [Step 0] Environment Setup & Dependencies Installation
!pip install -q google-genai google-adk pydantic requests
!nvidia-smi || echo "Running on CPU runtime"


In [ ]:
# [Step 0-1] Secure Credential Configuration (Colab Secrets / Userdata)
# Add your API key(s) (GEMINI_API_KEY) in Colab's left sidebar 'Secrets' (Key icon)
import os
for _key_name in ['GEMINI_API_KEY']:
    try:
        from google.colab import userdata
        _val = userdata.get(_key_name)
        if _val:
            os.environ[_key_name] = _val
            print(f"[OK] {_key_name} loaded securely from Colab Secrets.")
    except Exception:
        if not os.environ.get(_key_name):
            print(f"[INFO] {_key_name} not found in Colab Secrets (running in offline/fallback mode unless set).")


# Lab 01: Environment Verification, ADK Scaffolding & Custom Tools

## Objective
Verify your local environment using `./scripts/check_env.sh` and initialize your ADK agent using Google Agent Development Kit (`google-adk` / `google.golang.org/adk/v2` / `com.google.adk:google-adk` / `@google/adk`) with `gemini-4-hydrogen`.

## Official Multi-Language ADK Entry Points
- **Python (`google-adk`)**: `workshop/01_starter/sre_agent/agent.py` exposing `root_agent = LlmAgent(...)`
- **Go (`google.golang.org/adk/v2`)**: `llmagent.New`, `functiontool.New`, and `gemini.NewModel`
- **Java (`com.google.adk:google-adk`)**: `LlmAgent.builder()`, `SequentialAgent.builder()`, and `FunctionTool.create()`
- **Kotlin (`com.google.adk:google-adk-kotlin-core`)**: `LlmAgent(...)`, `Gemini("gemini-4-hydrogen")`, and `FunctionTool.create(::fn)`
- **TypeScript (`@google/adk`)**: `new LlmAgent({...})` and Zod-typed `new FunctionTool({...})`

## Instructions
1. Run `./scripts/check_env.sh` (or `.\scripts\check_env.ps1` on Windows).
2. If building with ADK & `agents-cli`, install the 7 official agent skills:
   ```bash
   uvx google-agents-cli setup
   ```
3. Open `workshop/01_starter/main.py` (and `workshop/01_starter/agent.py` or `polyglot/`) and inspect the starter tool definitions.
4. Test execution:
   ```bash
   python3 workshop/01_starter/main.py
   ```

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Starter Code for Enterprise Multi-Agent SRE with Polyglot ADK
Primary Cloud Provider: Google GenAI (GEMINI_API_KEY)
Primary Cloud Model: gemini-4-hydrogen | Local Ollama Model: gemma4:e4b
"""

import os
import json
from dataclasses import dataclass, asdict
from typing import List


@dataclass
class WorkshopTaskResult:
    status: str
    model_used: str
    summary: str
    action_items: List[str]


# =========================================================================
# TODO: [Lab 01 - Exercise] Initialize AI Client & Run Sanity Check
# Hint: Connect to Google GenAI (gemini-4-hydrogen via GEMINI_API_KEY) or Local Ollama (gemma4:e4b)
# =========================================================================
def lab1_sanity_check(prompt: str) -> str:
    return f"[Starter Lab 01 Placeholder] Received prompt: {prompt}"


# =========================================================================
# TODO: [Lab 02 - Exercise] Implement Structured Output Schema Parser
# Hint: Parse response into WorkshopTaskResult JSON schema
# =========================================================================
def lab2_structured_output(topic_name: str) -> WorkshopTaskResult:
    return WorkshopTaskResult(
        status="pending",
        model_used="gemini-4-hydrogen",
        summary=f"Starter placeholder for {topic_name}",
        action_items=["Complete Lab 01", "Complete Lab 02", "Verify Lab 03 pipeline"],
    )


def main():
    print("Welcome to the Enterprise Multi-Agent SRE with Polyglot ADK Workshop! Open workshop/03_labs/README.md to begin.")
    msg = lab1_sanity_check("Hello from adk-polyglot-workshop")
    print(msg)
    result = lab2_structured_output("Enterprise Multi-Agent SRE with Polyglot ADK")
    print(json.dumps(asdict(result), indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


# Lab 02: Multi-Agent Hierarchy, Sub-Agent Delegation & Structured Outputs

## Objective
Wire specialized sub-agents (`slo_specialist`, `report_specialist`) under a root coordinator (`root_agent` running `gemini-4-carbon`) and enforce deterministic JSON schema outputs.

## Instructions
1. Review `prompt-pack/README.md` for the target JSON Schema.
2. In Python (`workshop/01_starter/agent.py`), attach `sub_agents=[slo_specialist, report_specialist]` to `root_agent`.
3. If exploring the Go, Java, Kotlin, or TypeScript tracks under `workshop/02_final/polyglot/`, compare how `SubAgents` / `subAgents` and `SequentialAgent` delegate tasks across languages.
4. Launch the interactive ADK Dev UI to inspect sub-agent handoff events visually:
   ```bash
   cd workshop/02_final && adk web
   ```

In [ ]:
"""
ADK & Google Agents CLI (google-agents-cli) Starter Agent (`agent.py`)
Primary Provider: Google GenAI (gemini-4-hydrogen)
Compatible with:
  - `adk web` / `adk run` / `adk eval`
  - `uvx google-agents-cli setup`
  - `agents-cli eval dataset synthesize` / `agents-cli deploy`
"""
try:
    from google.adk.agents import LlmAgent
except ImportError:
    class LlmAgent:  # Offline fallback stub when google-adk is not yet installed
        def __init__(self, **kwargs):
            self.__dict__.update(kwargs)

def lookup_service_slo(service_name: str) -> dict:
    """Returns target SLO latency and remaining error budget for a production microservice.

    Args:
        service_name: Target microservice name (e.g. 'checkout-api', 'auth-service').
    """
    catalog = {
        "checkout-api": {"target_p99_ms": 250, "error_budget_remaining_pct": 42.0, "tier": "critical"},
        "auth-service": {"target_p99_ms": 120, "error_budget_remaining_pct": 88.5, "tier": "critical"},
    }
    return catalog.get(service_name, {"target_p99_ms": 500, "error_budget_remaining_pct": 100.0, "tier": "standard"})


# TODO [Lab 01]: Define specialist sub-agents and register them under `sub_agents=[...]` on `root_agent`
# Tip: For Anthropic Claude or OpenAI models in ADK, wrap with LiteLlm(model="anthropic/claude-sonnet-5-5") or LiteLlm(model="openai/gpt-6.1-sol")
root_agent = LlmAgent(
    name="adk_starter_coordinator",
    model="gemini-4-hydrogen",
    description="Starter ADK root_agent entry point for Enterprise Multi-Agent SRE with Polyglot ADK.",
    instruction="Use lookup_service_slo to answer questions about microservice SLOs.",
    tools=[lookup_service_slo],
)


# Lab 03: Trajectory Evaluation (adk eval), ADLC Optimization & Deployment

## Objective
Evaluate multi-agent tool trajectories against `evalset.json` (`adk eval` / `agents-cli eval`), run `InMemoryRunner` tests, and prepare for Cloud Run / Vertex AI Agent Engine deployment.

## Instructions
1. Run the reference solution:
   ```bash
   python3 workshop/02_final/main.py
   ```
2. Evaluate the ADK agent trajectory against `workshop/02_final/evalset.json`:
   ```bash
   adk eval workshop/02_final/sre_agent workshop/02_final/evalset.json
   ```
3. Run the automated workshop integrity verifier:
   ```bash
   python3 scripts/verify_workshop.py .
   ```

In [ ]:
# TODO [Lab 03]: Implement your code below
